In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize a trace
t = np.load('data/traces/0000.npy')
print(t)
plt.plot(t)

In [ ]:
# Parameter: ML-KEM-768
k  = 3
du = 10
dv = 4

# Load dk_pke
path_to_data = "data"
with open(f"{path_to_data}/dk", "rb") as f: dk = f.read()
dk_pke = dk[0 : 384*k]

In [ ]:
# Fixed constants
q = 3329
n = 256

# Below are helper functions for calculating intermediate value
def bytes_to_bits(b):
    l = len(b)
    a = bytearray(8*l)
    for i in range(0, 8*l, 8):
        x = b[i // 8]
        for j in range(8):
            a[i + j] = (x >> j) & 1
    return a

def byte_decode(d, b):
    if d < 12:
        m = 1 << d
    else:
        m = q
    b = bytes_to_bits(b)
    f = []
    for i  in range(256):
        x = 0
        for j in range(d):
            x += b[i*d + j] << j
        f += [ x % m ]
    return f

def decompress(d, yv):
    return [ (q*y + (1 << (d - 1))) >> d for y in yv ]

ML_KEM_ZETA_MUL = [
    17,     -17,    2761,   -2761,  583,    -583,   2649,   -2649,
    1637,   -1637,  723,    -723,   2288,   -2288,  1100,   -1100,
    1409,   -1409,  2662,   -2662,  3281,   -3281,  233,    -233,
    756,    -756,   2156,   -2156,  3015,   -3015,  3050,   -3050,
    1703,   -1703,  1651,   -1651,  2789,   -2789,  1789,   -1789,
    1847,   -1847,  952,    -952,   1461,   -1461,  2687,   -2687,
    939,    -939,   2308,   -2308,  2437,   -2437,  2388,   -2388,
    733,    -733,   2337,   -2337,  268,    -268,   641,    -641,
    1584,   -1584,  2298,   -2298,  2037,   -2037,  3220,   -3220,
    375,    -375,   2549,   -2549,  2090,   -2090,  1645,   -1645,
    1063,   -1063,  319,    -319,   2773,   -2773,  757,    -757,
    2099,   -2099,  561,    -561,   2466,   -2466,  2594,   -2594,
    2804,   -2804,  1092,   -1092,  403,    -403,   1026,   -1026,
    1143,   -1143,  2150,   -2150,  2775,   -2775,  886,    -886,
    1722,   -1722,  1212,   -1212,  1874,   -1874,  1029,   -1029,
    2110,   -2110,  2935,   -2935,  885,    -885,   2154,   -2154 ]

ML_KEM_ZETA_NTT = [
    1,      1729,   2580,   3289,   2642,   630,    1897,   848,
    1062,   1919,   193,    797,    2786,   3260,   569,    1746,
    296,    2447,   1339,   1476,   3046,   56,     2240,   1333,
    1426,   2094,   535,    2882,   2393,   2879,   1974,   821,
    289,    331,    3253,   1756,   1197,   2304,   2277,   2055,
    650,    1977,   2513,   632,    2865,   33,     1320,   1915,
    2319,   1435,   807,    452,    1438,   2868,   1534,   2402,
    2647,   2617,   1481,   648,    2474,   3110,   1227,   910,
    17,     2761,   583,    2649,   1637,   723,    2288,   1100,
    1409,   2662,   3281,   233,    756,    2156,   3015,   3050,
    1703,   1651,   2789,   1789,   1847,   952,    1461,   2687,
    939,    2308,   2437,   2388,   733,    2337,   268,    641,
    1584,   2298,   2037,   3220,   375,    2549,   2090,   1645,
    1063,   319,    2773,   757,    2099,   561,    2466,   2594,
    2804,   1092,   403,    1026,   1143,   2150,   2775,   886,
    1722,   1212,   1874,   1029,   2110,   2935,   885,    2154 ]

def ntt(f):
    f   = f.copy()
    i   = 1
    le  = 128
    while le >= 2:
        for st in range(0, 256, 2*le):
            ze = ML_KEM_ZETA_NTT[i]
            i += 1
            for j in range(st, st + le):
                t           = (ze*f[j + le]) % q
                f[j + le]   = (f[j] - t) % q
                f[j]        = (f[j] + t) % q
        le //= 2
    return f

def base_case_multiply(a0, a1, b0, b1, gam):
    c0  = (a0*b0 + a1*b1*gam) % q
    c1  = (a0*b1 + a1*b0) % q
    return [ c0, c1 ]

def multiply_ntts(f, g):
    h = []
    for ii in range(0, 256, 2):
        h += base_case_multiply(f[ii], f[ii+1], g[ii], g[ii+1],
                                        ML_KEM_ZETA_MUL[ii//2])
    return h

def poly_add(f, g):
    return [ (f[i] + g[i]) % q for i in range(256) ]

In [ ]:
# Load traces and ciphertexts from files

import os
trace_files = [os.path.join(f"{path_to_data}/traces", f) 
                for f in os.listdir(f"{path_to_data}/traces") if ".npy" in f]
trace_files = sorted(trace_files)
traces = [np.load(f) for f in trace_files]

print(f"Total number of traces      : {len(traces)}")
print(f"Number of samples per trace : {len(traces[0])}")

ciphertext_files = [os.path.join(f"{path_to_data}/ciphertexts", f) 
                for f in os.listdir(f"{path_to_data}/ciphertexts") if ".npy" in f]
ciphertext_files = sorted(ciphertext_files)
ciphertexts = [np.load(f) for f in ciphertext_files]

print(f"Total number of ciphertexts      : {len(ciphertexts)}")
print(f"Number of samples per ciphertext : {len(ciphertexts[0])}")

In [ ]:
# This is the expected secret key (to be recovered).
# It is used for 2 purposes:
# - Verify if the recovered key is correct
# - In the attack, we need to guess 2 coefficients of the key, thus 3329*3329 possibilities.
# This is a huge number for a classical computer. So we assume 1 coefficient is know, and guess
# only 1 coefficient for convenience.
s_expected  = [byte_decode(12, dk_pke[384*i:384*(i+1)]) for i in range(k) ]
for i in range(k): print(s_expected[i])

In [ ]:
# These two values are calculated once and used for different predictions
mean_t = np.mean(traces, axis=0)
diff_t = traces - mean_t

def calculate_correlation(diff_t, predictions):
    mean_p = np.mean(predictions)
    diff_p = predictions - mean_p

    sum_t_sq = np.sum(diff_t**2, axis=0)
    sum_p_sq = np.sum(diff_p**2)

    nominator = np.sum(diff_t * diff_p[:, None], axis=0)

    # Silence the divide-by-zero warnings
    with np.errstate(divide='ignore', invalid='ignore'):
        denominator  = np.sqrt(sum_t_sq * sum_p_sq)
        correlations = nominator / denominator

    # Replace any NaN (caused by 0/0) with 0
    correlations = np.nan_to_num(correlations)
    
    return correlations


In [ ]:
def recover_key(expected_key, poly_index, coef_index):
    '''
    :param expected_key: correct key used to compared with recovered key
    :param poly_index  : in range(0, k), which polynomial in the secret key
    :param coef_index  : in range(0, q), which coefficient to recover
    '''
    i = poly_index
    j = coef_index
    if i not in range(0, k): raise ValueError("poly_index must be in range(0, k)")
    if j % 2 != 0: raise ValueError("Only support recovering coefficient at even indexes")
    si = expected_key[i]

    best_guess = None
    corr_trace = None
    corr_max   = 0
    for guess in range(q):
        # For hypothetical power consumption
        predictions = []

        for c in ciphertexts:
            c1 = c[0 : 32*du*k]
            # c2 = c[32*du*k : 32*(du*k + dv)]
            upi = decompress(du, byte_decode(du, c1[32*du*i : 32*du*(i+1)]))
            upi = ntt(upi)

            # The first output of base_case_multiply is chosen as the intermediate value. In practice,
            # we need to guess 2 coefficients of the key, thus 3329*3329 possibilities. However, we assume
            # the second coefficient is known to reduce the search space in this demo.
            # 
            # We can alternatively target the multiplication a0*b0 in order to guess only 1 coefficient.
            # But this approach will cause some group of key candidates which yield the same correlation
            # with the power traces.
            intermediate, _ = base_case_multiply(guess, si[j+1], upi[j], upi[j+1], ML_KEM_ZETA_MUL[j//2])

            # hypothetical power
            hw = bin(intermediate).count('1')
            predictions.append(hw)

        correlations = calculate_correlation(diff_t, predictions)
        corr_max_local = np.max(np.abs(correlations))
        print(f'Guess {guess:4d}: {corr_max_local:.4f}')
        if corr_max_local > corr_max:
            best_guess = guess
            corr_trace = correlations
            corr_max   = corr_max_local

    if best_guess == si[j]:
        print(f'Bravo! Recovered {best_guess} - Expected {si[j]}')
    else:
        print(f'Incorrect! :( Recovered {best_guess} - Expected {si[j]}')
    
    return best_guess, corr_trace

In [ ]:
poly_index = 1
coef_index = 10
best_guess, corr_trace = recover_key(s_expected, poly_index, coef_index)

In [ ]:
plt.plot(corr_trace)